# Masterclass 1: Multi-Sector Exploratory Spatial Data Analysis (ESDA) & Disease Surveillance
### *Evidence-Based Spatial Intelligence Across Public Health, Disease Epidemiology (Malaria), Geomarketing, Cultural Geography, and Infrastructure*

---

## 1. Introduction & Key Concepts

In classical non-spatial data science, observations are assumed to be **independent and identically distributed (i.i.d.)**. When analyzing geographic units—such as Nigeria's **9,308 administrative wards**—this assumption fundamentally collapses due to **Tobler's First Law of Geography**:

> *"Everything is related to everything else, but near things are more related than distant things."*  
> — Waldo Tobler (1970)

Spatial autocorrelation arises naturally: pathogens (e.g. *Plasmodium falciparum* malaria vectors), trade corridors, and cultural traditions cross administrative borders. Ignoring spatial dependence leads to:
1. **Underestimated Standard Errors:** Artificially deflated variances causing false statistical significance (**Type-I Error**).
2. **The Spatial Data Leakage Trap in ML:** Random train/test splits leak neighboring information, producing high test accuracy that fails upon field deployment.
3. **Misallocated Capital:** Opening clinics or commercial retail branches without accounting for neighborhood catchment and spillovers.

### Learning Objectives:
- **Spatial Topology & Weights Matrix ($W$):** Queen contiguity vs. $K$-Nearest Neighbors ($k=5$) and row-standardization ($w_{ij}^*$).
- **Global Spatial Autocorrelation (Moran's $I$):** Formal permutation testing of spatial clustering vs. complete spatial randomness ($H_0$).
- **Local Indicators of Spatial Association (LISA / Anselin Local Moran's $I_i$):** Pinpointing statistically significant **Hotspots ($HH$)**, **Coldspots ($LL$)**, and **Spatial Outliers ($HL, LH$)**.
- **Cross-Sector Visual Analytics:** National choropleth maps with explicit legends, correlation heatmaps, Moran scatterplots, and 6-panel infrastructure galleries.
- **Disease Surveillance Deep Dive:** Modeling ward-level **Malaria Parasite Prevalence ($Pf	ext{PR}_{2-10}$)** and spatial transmission clusters.



## 2. Mathematical Framework & Formal Definitions

### 2.1 The Spatial Weights Matrix ($W$) & Spatial Lag ($Wy$)
Let $S = \{1, 2, \dots, n\}$ be the set of $n$ spatial wards. A spatial weights matrix $W$ is an $n \times n$ matrix where entry $w_{ij}$ quantifies the spatial relationship between ward $i$ and ward $j$:

$$
w_{ij} = 
\begin{cases} 
1 & \text{if } j \in N(i) \text{ and } i \neq j \\ 
0 & \text{otherwise} 
\end{cases}
$$

Row-standardization ensures scale invariance regardless of neighbor counts:

$$
w_{ij}^* = \frac{w_{ij}}{\sum_{k=1}^n w_{ik}} \quad \implies \quad \sum_{j=1}^n w_{ij}^* = 1
$$

The **Spatial Lag** $[Wy]_i$ calculates the spatially weighted neighborhood average:

$$
[Wy]_i = \sum_{j=1}^n w_{ij}^* y_j
$$

---

### 2.2 Global Moran's $I$
Global Moran's $I$ tests for overall spatial clustering across the entire national study area:

$$
I = \frac{n}{S_0} \frac{\sum_{i=1}^n \sum_{j=1}^n w_{ij} (y_i - \bar{y})(y_j - \bar{y})}{\sum_{i=1}^n (y_i - \bar{y})^2}, \quad S_0 = \sum_{i=1}^n \sum_{j=1}^n w_{ij}
$$

- **Expected Value under Null Hypothesis $H_0$ (Spatial Randomness):**
  $$E[I] = -\frac{1}{n-1} \xrightarrow{n \to \infty} 0$$
- **Interpretation:** $I > E[I]$ ($p < 0.05$) indicates **Positive Spatial Autocorrelation** (Clustering of similar values).

---

### 2.3 Local Indicators of Spatial Association (LISA)
Decomposes global autocorrelation into local ward-level statistics:

$$
I_i = \frac{z_i}{s^2} \sum_{j=1}^n w_{ij} z_j, \quad z_i = y_i - \bar{y}, \quad s^2 = \frac{1}{n}\sum_{i=1}^n z_i^2
$$

The four quadrants:
1. **High-High ($HH$ - Hotspot):** High focal value surrounded by high neighbor values.
2. **Low-Low ($LL$ - Coldspot):** Low focal value surrounded by low neighbor values.
3. **High-Low ($HL$ - Outlier):** High focal value surrounded by low neighbor values ("Island of Wealth/Protection").
4. **Low-High ($LH$ - Outlier):** Low focal value surrounded by high neighbor values ("Pocket of Deprivation/Vulnerability").


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import libpysal
import esda
from splot.esda import plot_moran, moran_scatterplot, lisa_cluster

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
sns.set_style('whitegrid')


In [ ]:
DATA_PATH = '../data/processed/nigeria_wards_master.parquet'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/nigeria_wards_master.parquet'

print(f"Loading master spatial dataset from: {DATA_PATH}")
gdf = gpd.read_parquet(DATA_PATH)

if gdf.crs is None:
    gdf.set_crs(epsg=4326, inplace=True)
else:
    gdf = gdf.to_crs(epsg=4326)

gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
gdf.reset_index(drop=True, inplace=True)

# Continuous feature imputation
gdf['rwi_mean'] = gdf['rwi_mean'].fillna(gdf['rwi_mean'].median())
gdf['pop_2025_sum'] = gdf['pop_2025_sum'].fillna(gdf['pop_2025_sum'].median())
gdf['population_density_per_sqkm'] = gdf['population_density_per_sqkm'].fillna(gdf['population_density_per_sqkm'].median())

# Rates per 10,000 population
gdf['health_facility_rate'] = (gdf['health_facilities_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['is_health_desert'] = (gdf['health_facilities_count'] == 0) & (gdf['pop_2025_sum'] > gdf['pop_2025_sum'].median())
gdf['market_rate'] = (gdf['markets_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['water_point_rate'] = (gdf['water_points_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['police_station_rate'] = (gdf['police_stations_count'] / (gdf['pop_2025_sum'] + 100)) * 10000

# Cultural Geography Metrics
gdf['total_faith_institutions'] = gdf['churches_count'] + gdf['mosques_count']
gdf['church_share'] = gdf['churches_count'] / (gdf['total_faith_institutions'] + 1e-5)
gdf['mosque_share'] = gdf['mosques_count'] / (gdf['total_faith_institutions'] + 1e-5)
p1 = gdf['church_share'].clip(lower=1e-5, upper=1-1e-5)
p2 = gdf['mosque_share'].clip(lower=1e-5, upper=1-1e-5)
gdf['religious_diversity_idx'] = -(p1 * np.log2(p1) + p2 * np.log2(p2))

# Commercial Retail Segments
rwi_med = gdf['rwi_mean'].median()
mkt_med = gdf['markets_count'].median()
conditions = [
    (gdf['rwi_mean'] >= rwi_med) & (gdf['markets_count'] >= mkt_med),
    (gdf['rwi_mean'] >= rwi_med) & (gdf['markets_count'] < mkt_med),
    (gdf['rwi_mean'] < rwi_med) & (gdf['markets_count'] >= mkt_med),
    (gdf['rwi_mean'] < rwi_med) & (gdf['markets_count'] < mkt_med)
]
choices = [
    "Tier 1: Saturated Affluent (High Wealth, High Markets)",
    "Tier 2: Prime Expansion Target (High Wealth, Low Markets)",
    "Tier 3: Informal Commerce Hubs (Low Wealth, High Markets)",
    "Tier 4: Subsistence / Underdeveloped (Low Wealth, Low Markets)"
]
gdf['retail_segment'] = np.select(conditions, choices, default='Unclassified')

print(f"Loaded {len(gdf):,} valid administrative wards.")


## 3. Multi-Sector Descriptive Statistics & Correlation Analysis

We examine the baseline distributions across all key sectors before running spatial tests.


In [ ]:
sector_cols = [
    'rwi_mean', 'pop_2025_sum', 'health_facility_rate', 
    'market_rate', 'water_point_rate', 'police_station_rate',
    'churches_count', 'mosques_count'
]
if 'malaria_prevalence_pct' in gdf.columns:
    sector_cols.append('malaria_prevalence_pct')

summary_df = gdf[sector_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
summary_df['skewness'] = gdf[sector_cols].skew()
print("=== MULTI-SECTOR STATISTICAL DISTRIBUTIONS (9,308 WARDS) ===")
summary_df[['mean', 'std', '5%', '50%', '95%', 'skewness']].round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = gdf[sector_cols].corr(method='spearman')
sns.heatmap(corr, annot=True, fmt=".2f", cmap='Spectral_r', vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={'label': 'Spearman Rank Correlation'})
ax.set_title("Cross-Sector Non-Parametric Correlation Matrix", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


## 4. National Multi-Sector Infrastructure Density Gallery

Below, we display the geographic distribution across six fundamental infrastructure and demographic layers, each with an explicit, labeled colorbar legend:
1. **Health Facility Rate** (Clinics per 10k residents)
2. **Commercial Market Rate** (Markets per 10k residents)
3. **Public Water Point Rate** (WASH points per 10k residents)
4. **Church Density** (Christian institutional footprints)
5. **Mosque Density** (Islamic institutional footprints)
6. **Population Density** (Persons per km²)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

gallery_layers = [
    ('health_facility_rate', 'Health Facilities per 10k Pop', 'Reds', 'Facilities / 10k'),
    ('market_rate', 'Markets per 10k Pop', 'Greens', 'Markets / 10k'),
    ('water_point_rate', 'Water Points per 10k Pop', 'Blues', 'Water Points / 10k'),
    ('churches_count', 'Churches Count per Ward', 'Purples', 'Churches Count'),
    ('mosques_count', 'Mosques Count per Ward', 'Oranges', 'Mosques Count'),
    ('population_density_per_sqkm', 'Population Density (per km²)', 'viridis', 'Persons / sq km')
]

for ax, (col, title, cmap, lbl) in zip(axes, gallery_layers):
    # Clip extreme 2% outliers for high-contrast visual display
    p98 = gdf[col].quantile(0.98)
    sub_plot = gdf.assign(val=gdf[col].clip(upper=p98))
    sub_plot.plot(column='val', cmap=cmap, ax=ax, legend=True,
                 legend_kwds={'label': lbl, 'orientation': 'horizontal', 'shrink': 0.65, 'pad': 0.05})
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()


## 5. Public Health Spotlight: Detecting "Healthcare Deserts"

### The Decision Challenge:
Healthcare Deserts are defined as administrative wards where the population exceeds the national median (> 17,000 residents) but has **exactly 0 registered health facilities**.
- Bypassing state quotas to route mobile clinics and maternal care centers directly to high-risk populations.


In [ ]:
n_deserts = gdf['is_health_desert'].sum()
desert_pop = gdf[gdf['is_health_desert']]['pop_2025_sum'].sum()

print(f"Total Healthcare Desert Wards: {n_deserts:,} ({n_deserts/len(gdf)*100:.2f}%)")
print(f"Vulnerable Population in Deserts: {desert_pop:,.0f} people")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))

# Map 1: RWI
gdf.plot(column='rwi_mean', cmap='viridis', legend=True, ax=ax1,
         legend_kwds={'label': 'Relative Wealth Index (RWI)', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax1.set_title("A. National Relative Wealth Index (RWI)", fontsize=12, fontweight='bold')
ax1.axis('off')

# Map 2: Healthcare Deserts with explicit patch legend
gdf.plot(color='#ececec', edgecolor='#ffffff', linewidth=0.1, ax=ax2)
gdf[gdf['is_health_desert']].plot(color='#d90429', ax=ax2)
ax2.set_title(f"B. Critical Healthcare Deserts (n={n_deserts:,})", fontsize=12, fontweight='bold')
ax2.axis('off')

desert_p = mpatches.Patch(color='#d90429', label=f"Healthcare Deserts (n={n_deserts:,})")
base_p = mpatches.Patch(color='#ececec', label=f"Wards with Health Facilities (n={(~gdf['is_health_desert']).sum():,})")
ax2.legend(handles=[desert_p, base_p], loc='lower left', frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()


## 6. Epidemiological Disease Surveillance: Malaria Transmission

Vector-borne diseases like malaria do not respect ward boundaries. We map:
1. **Malaria Parasite Prevalence ($Pf	ext{PR}_{2-10}$ %):** High in humid southern mangrove/rainforest basins, seasonal in northern sahel.
2. **Anselin Local Moran Disease Clusters:** Distinguishing endemic transmission hotspots from protected zones.


In [ ]:
if 'malaria_prevalence_pct' in gdf.columns:
    w_knn = libpysal.weights.KNN.from_dataframe(gdf, k=5)
    w_knn.transform = 'R'
    
    lm_mal = esda.Moran_Local(gdf['malaria_prevalence_pct'].values, w_knn, permutations=999, seed=42)
    sig_m = lm_mal.p_sim < 0.05
    gdf['malaria_lisa'] = 'Not Significant'
    gdf.loc[(lm_mal.q == 1) & sig_m, 'malaria_lisa'] = 'High-High (Endemic Hotspot)'
    gdf.loc[(lm_mal.q == 3) & sig_m, 'malaria_lisa'] = 'Low-Low (Low Transmission)'
    gdf.loc[(lm_mal.q == 4) & sig_m, 'malaria_lisa'] = 'High-Low (Outlier Pocket)'
    gdf.loc[(lm_mal.q == 2) & sig_m, 'malaria_lisa'] = 'Low-High (Protected Pocket)'

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))
    
    # Map 1: Continuous Malaria Rate
    gdf.plot(column='malaria_prevalence_pct', cmap='YlOrRd', legend=True, ax=ax1,
             legend_kwds={'label': 'Malaria Parasite Prevalence (PfPR 2-10 %)', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
    ax1.set_title("A. Malaria Parasite Prevalence Across Nigeria", fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    # Map 2: LISA Clusters
    mal_colors = {
        'Not Significant': '#f0f0f0',
        'High-High (Endemic Hotspot)': '#d90429',
        'Low-Low (Low Transmission)': '#0077b6',
        'High-Low (Outlier Pocket)': '#f77f00',
        'Low-High (Protected Pocket)': '#90e0ef'
    }
    for ctype, color in mal_colors.items():
        sub = gdf[gdf['malaria_lisa'] == ctype]
        if len(sub) > 0:
            sub.plot(color=color, ax=ax2, linewidth=0.1, edgecolor='white')
    
    ax2.set_title("B. Anselin LISA Malaria Transmission Clusters", fontsize=12, fontweight='bold')
    ax2.axis('off')
    mal_patches = [mpatches.Patch(color=c, label=f"{l} (n={(gdf['malaria_lisa'] == l).sum():,})") for l, c in mal_colors.items()]
    ax2.legend(handles=mal_patches, loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
    
    plt.tight_layout()
    plt.show()


## 7. Commercial Geomarketing: Retail Catchment Optimization

Segmenting wards into four strategic commercial quadrants:
- **Tier 2 (Prime Expansion Targets):** High Relative Wealth Index (RWI) but Low Market Density.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))

segment_colors = {
    "Tier 1: Saturated Affluent (High Wealth, High Markets)": "#1b4332",
    "Tier 2: Prime Expansion Target (High Wealth, Low Markets)": "#52b788",
    "Tier 3: Informal Commerce Hubs (Low Wealth, High Markets)": "#e76f51",
    "Tier 4: Subsistence / Underdeveloped (Low Wealth, Low Markets)": "#d8d8d8"
}

for seg, col in segment_colors.items():
    subset = gdf[gdf['retail_segment'] == seg]
    subset.plot(color=col, ax=ax1, linewidth=0.1, edgecolor='white')

ax1.set_title("A. Commercial Catchment & Retail Strategy Map", fontsize=12, fontweight='bold')
ax1.axis('off')
seg_patches = [mpatches.Patch(color=c, label=l) for l, c in segment_colors.items()]
ax1.legend(handles=seg_patches, loc='lower left', fontsize=8.5, frameon=True, facecolor='white', framealpha=0.9)

sns.countplot(data=gdf, y='retail_segment', hue='retail_segment', palette=list(segment_colors.values()), order=choices, ax=ax2, legend=False)
ax2.set_title("B. Distribution of Wards Across Retail Strategy Tiers", fontsize=12, fontweight='bold')
ax2.set_xlabel("Number of Administrative Wards")
ax2.set_ylabel("")

plt.tight_layout()
plt.show()


## 8. Cultural Geography: Faith Institutions & Shannon Entropy Diversity

Mapping institutional sorting and cultural transition zones across Nigeria's geopolitical landscape.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))

gdf.plot(column='church_share', cmap='coolwarm', legend=True, ax=ax1,
         legend_kwds={'label': 'Institutional Proportion (0=Mosque Dominant, 1=Church Dominant)', 
                      'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax1.set_title("A. Faith Institution Composition (Churches vs Mosques)", fontsize=12, fontweight='bold')
ax1.axis('off')

gdf.plot(column='religious_diversity_idx', cmap='magma', legend=True, ax=ax2,
         legend_kwds={'label': 'Shannon Entropy Diversity Index', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax2.set_title("B. Cultural Transition Zones (High Co-Presence)", fontsize=12, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()


## 9. Spatial Topology: Constructing Spatial Weights ($W$)

We build row-standardized $K$-Nearest Neighbors ($k=5$) weights for all 9,308 wards.


In [ ]:
w_knn = libpysal.weights.KNN.from_dataframe(gdf, k=5)
w_knn.transform = 'R'
sparsity = (1.0 - (w_knn.nonzero / (w_knn.n ** 2))) * 100
print(f"Spatial Weights W: n={w_knn.n}, Mean Neighbors={w_knn.mean_neighbors:.1f}, Sparsity={sparsity:.2f}%")


## 10. Global Moran's $I$ Hypothesis Testing Across All Sectors

Permutation tests (999 iterations) evaluate the hypothesis of spatial randomness ($H_0$).


In [ ]:
test_vars = {
    'Wealth (RWI Mean)': 'rwi_mean',
    'Health Facilities Rate': 'health_facility_rate',
    'Commercial Market Rate': 'market_rate',
    'Water Points Rate': 'water_point_rate',
    'Church Proportion': 'church_share',
    'Population Density': 'population_density_per_sqkm'
}
if 'malaria_prevalence_pct' in gdf.columns:
    test_vars['Malaria Prevalence Rate'] = 'malaria_prevalence_pct'

moran_records = []
for label, col in test_vars.items():
    mi = esda.Moran(gdf[col].values, w_knn, permutations=999)
    moran_records.append({
        'Sector / Variable': label,
        "Moran's I": round(mi.I, 4),
        "Expected E[I]": round(mi.EI, 5),
        "z-score": round(mi.z_sim, 2),
        "p-value": mi.p_sim,
        "Spatial Pattern": "Strong Clustering" if mi.I > 0.4 else "Moderate Clustering"
    })

moran_table = pd.DataFrame(moran_records)
print("=== GLOBAL MORAN'S I TEST RESULTS ===")
moran_table


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))

mi_rwi = esda.Moran(gdf['rwi_mean'].values, w_knn)
moran_scatterplot(mi_rwi, ax=ax1)
ax1.set_title(f"A. Moran Scatterplot: Wealth (RWI)\nMoran's I = {mi_rwi.I:.3f}, z = {mi_rwi.z_sim:.1f}", fontsize=12, fontweight='bold')
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.axvline(0, color='gray', linestyle='--', linewidth=0.8)

target_col = 'malaria_prevalence_pct' if 'malaria_prevalence_pct' in gdf.columns else 'health_facility_rate'
target_lbl = 'Malaria Prevalence Rate' if 'malaria_prevalence_pct' in gdf.columns else 'Health Facility Rate'
mi_target = esda.Moran(gdf[target_col].values, w_knn)
moran_scatterplot(mi_target, ax=ax2)
ax2.set_title(f"B. Moran Scatterplot: {target_lbl}\nMoran's I = {mi_target.I:.3f}, z = {mi_target.z_sim:.1f}", fontsize=12, fontweight='bold')
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.axvline(0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()


## 11. Anselin Local Moran's $I_i$ (LISA): National Wealth Clusters

Isolating statistically significant hotspots, coldspots, and spatial outliers across Nigeria.


In [ ]:
lm_rwi = esda.Moran_Local(gdf['rwi_mean'].values, w_knn, transformation='r', permutations=999, seed=42)
sig = lm_rwi.p_sim < 0.05

gdf['wealth_lisa'] = 'Not Significant'
gdf.loc[(lm_rwi.q == 1) & sig, 'wealth_lisa'] = 'High-High (Hotspot)'
gdf.loc[(lm_rwi.q == 3) & sig, 'wealth_lisa'] = 'Low-Low (Coldspot)'
gdf.loc[(lm_rwi.q == 4) & sig, 'wealth_lisa'] = 'High-Low (Outlier)'
gdf.loc[(lm_rwi.q == 2) & sig, 'wealth_lisa'] = 'Low-High (Outlier)'

fig, ax = plt.subplots(figsize=(11, 8.5))
lisa_colors = {
    'Not Significant': '#f0f0f0',
    'High-High (Hotspot)': '#d90429',
    'Low-Low (Coldspot)': '#0077b6',
    'High-Low (Outlier)': '#f77f00',
    'Low-High (Outlier)': '#90e0ef'
}

for ctype, color in lisa_colors.items():
    sub = gdf[gdf['wealth_lisa'] == ctype]
    if len(sub) > 0:
        sub.plot(color=color, ax=ax, linewidth=0.1, edgecolor='white')

ax.set_title("Anselin Local Moran's I (LISA) Wealth Cluster Map", fontsize=13, fontweight='bold')
ax.axis('off')
lisa_patches = [mpatches.Patch(color=c, label=f"{l} (n={(gdf['wealth_lisa'] == l).sum():,})") for l, c in lisa_colors.items()]
ax.legend(handles=lisa_patches, loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9.5)

plt.tight_layout()
plt.show()


## 12. Strategic Multi-Sector Decision Playbook

| Spatial Quadrant | Public Health Action | Commercial Geomarketing Action | Disease & Vector Control Action |
| :--- | :--- | :--- | :--- |
| **High-High (Hotspot)** | Private hospital licensing; premium health maintenance organizations (HMOs). | Flagship retail stores, modern supermarkets, agency banking networks. | Routine urban surveillance; indoor pest management. |
| **Low-Low (Coldspot)** | Subsidized primary health clinics, mobile clinical vans, free maternal kits. | Low-ticket consumer products, sachet goods, micro-finance depots. | Intensive Indoor Residual Spraying (IRS) and universal bed net distribution. |
| **High-Low (Outlier)** | Regional referral hospital; serves as medical hub for surrounding rural areas. | Regional wholesale depot, cash-and-carry warehouse. | Regional diagnostic laboratory and antimalarial medication stockpile. |
| **Low-High (Outlier)** | Municipal utility expansion; connecting excluded slum communities to city care. | Community retail shops, commuter transit sales points. | Environmental drainage remediation and larvicide application. |
